# task #4.1 data sanitization

**based on**: Morris Chang, Di Zhuang, G. Dumindu Samaraweera - *Privacy-Preserving Machine Learning* (2023, Manning)

**dataset**: the titanic test file `deepeshkansotia/hantavirus-transmission-and-risk-datasete`.

the four techniques:
- **generalization** - replace a precise value with a more general one (a range, or a coarser group).
- **suppression** - remove an identifying record/column entirely.
- **perturbation** - replace values with random data that keeps the same statistical distribution.
- **anatomization** - split sensitive attributes and quasi-identifiers into two separate tables.

for each technique we show a sample of the resulting dataset and a short explanation.

mapping the techniques to this dataset's columns:
- `case_id` and `region` are (near) unique direct identifiers -> good suppression targets.
- `patient_age`, `country`, `gender` are quasi-identifiers -> generalization / perturbation.
- `virus_strain`, `fatality`, `hospitalization`, `symptoms` are sensitive medical attributes
  -> kept but separated by anatomization.


## 1. import and understand dataset

In [4]:
import sys
!{sys.executable} -m pip install sklearn-pandas kagglehub --quiet

import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

import kagglehub, os
path = kagglehub.dataset_download("deepeshkansotia/hantavirus-transmission-and-risk-dataset")
files = os.listdir(path)
df = pd.read_csv(os.path.join(path, [f for f in files if f.endswith(".csv")][0]))
print("shape:", df.shape)
df.head()

Using Colab cache for faster access to the 'hantavirus-transmission-and-risk-dataset' dataset.
shape: (2000, 19)


,case_id,country,region,report_date,virus_strain,transmission_type,exposure_source,patient_age,gender,symptoms,hospitalization,fatality,recovery_days,temperature_celsius,humidity_percent,rodent_presence_index,quarantine_days,population_density,air_quality_index
0,HV2026_00001,Canada,Lake Rickyside,24-11-2025,Sin Nombre,Rodent-to-Human,Agricultural Exposure,26,Male,"Headache, Vomiting, Fever",Yes,No,22.0,27.0,78.9,7,1,538,67
1,HV2026_00002,Bolivia,East Crystal,22-04-2025,Sin Nombre,Human-to-Human,Home Infestation,41,Female,"Fever, Muscle Pain, Fatigue",Yes,No,41.0,23.2,70.9,3,13,5624,162
2,HV2026_00003,Chile,Rossmouth,21-03-2025,Seoul,Human-to-Human,Home Infestation,39,Female,"Fever, Muscle Pain, Fatigue",No,No,31.0,22.6,38.5,8,17,2095,213
3,HV2026_00004,Argentina,West Ryan,10-02-2025,Sin Nombre,Human-to-Human,Rodent Exposure,49,Male,"Fever, Muscle Pain, Fatigue",No,No,11.0,22.6,87.0,7,8,7478,206
4,HV2026_00005,Chile,Mariaberg,25-03-2025,Dobrava,Rodent-to-Human,Home Infestation,59,Male,"Fever, Cough, Headache",No,No,24.0,33.5,36.2,8,12,4472,132


In [5]:
# keep a copy of the original to compare against after each change
original_df = df.copy()
print("columns:", list(df.columns))

columns: ['case_id', 'country', 'region', 'report_date', 'virus_strain', 'transmission_type', 'exposure_source', 'patient_age', 'gender', 'symptoms', 'hospitalization', 'fatality', 'recovery_days', 'temperature_celsius', 'humidity_percent', 'rodent_presence_index', 'quarantine_days', 'population_density', 'air_quality_index']


In [6]:
for i in df.columns.tolist():
  print("\nCOLUMN NAME\n", i, "\nDATA TYPE\n", df[i].dtype, "\nUNIQUE VALUES\n", df[i].unique(), "\nVALUE COUNTS\n", df[i].value_counts())
  print("________________________________________________")


COLUMN NAME
 case_id 
DATA TYPE
 object 
UNIQUE VALUES
 ['HV2026_00001' 'HV2026_00002' 'HV2026_00003' ... 'HV2026_01998'
 'HV2026_01999' 'HV2026_02000'] 
VALUE COUNTS
 case_id
HV2026_01984    1
HV2026_01983    1
HV2026_01982    1
HV2026_01981    1
HV2026_01980    1
               ..
HV2026_00005    1
HV2026_00004    1
HV2026_00003    1
HV2026_00002    1
HV2026_00001    1
Name: count, Length: 2000, dtype: int64
________________________________________________

COLUMN NAME
 country 
DATA TYPE
 object 
UNIQUE VALUES
 ['Canada' 'Bolivia' 'Chile' 'Argentina' 'Peru' 'Uruguay' 'USA' 'Brazil'
 'Mexico' 'Paraguay'] 
VALUE COUNTS
 country
Bolivia      215
Argentina    211
Canada       208
USA          208
Brazil       207
Uruguay      202
Mexico       196
Paraguay     193
Chile        190
Peru         170
Name: count, dtype: int64
________________________________________________

COLUMN NAME
 region 
DATA TYPE
 object 
UNIQUE VALUES
 ['Lake Rickyside' 'East Crystal' 'Rossmouth' ... 'Cowanbury'


In [18]:
print("df['regio'].nunique()   ", df["region"].nunique())
print("df['case_id'].nunique() ", df["case_id"].nunique())

df['regio'].nunique()    1850
df['case_id'].nunique()  2000


## 2. suppression
suppression removes identifying records or whole columns from the dataset (the book does
`df.drop(columns=[...])`). here `case_id` is a unique per-patient id and `region` has 1850
unique values out of 2000 rows, so both are effectively direct identifiers. we drop them so
a published copy can't be tied back to an individual by id or precise locality.

In [22]:
suppressed_df = df.drop(columns=["case_id", "region"])
print("before:", df.shape, "\nafter suppression:", suppressed_df.shape)
suppressed_df.head()

before: (2000, 19) 
after suppression: (2000, 17)


,country,report_date,virus_strain,transmission_type,exposure_source,patient_age,gender,symptoms,hospitalization,fatality,recovery_days,temperature_celsius,humidity_percent,rodent_presence_index,quarantine_days,population_density,air_quality_index
0,Canada,24-11-2025,Sin Nombre,Rodent-to-Human,Agricultural Exposure,26,Male,"Headache, Vomiting, Fever",Yes,No,22.0,27.0,78.9,7,1,538,67
1,Bolivia,22-04-2025,Sin Nombre,Human-to-Human,Home Infestation,41,Female,"Fever, Muscle Pain, Fatigue",Yes,No,41.0,23.2,70.9,3,13,5624,162
2,Chile,21-03-2025,Seoul,Human-to-Human,Home Infestation,39,Female,"Fever, Muscle Pain, Fatigue",No,No,31.0,22.6,38.5,8,17,2095,213
3,Argentina,10-02-2025,Sin Nombre,Human-to-Human,Rodent Exposure,49,Male,"Fever, Muscle Pain, Fatigue",No,No,11.0,22.6,87.0,7,8,7478,206
4,Chile,25-03-2025,Dobrava,Rodent-to-Human,Home Infestation,59,Male,"Fever, Cough, Headache",No,No,24.0,33.5,36.2,8,12,4472,132


the book applies the techniques in sequence: suppress -> generalize -> perturb

## 3. generalization
generalization replaces a precise value with a coarser one. the book shows two forms:
- **categorical**: encode/group categories - `LabelEncoder` via `DataFrameMapper` on `gender` and `country` so the exact label is hidden.
- **numerical**: replace an exact number with a *range* - `patient_age` into 10-year buckets.

In [26]:
from sklearn_pandas import DataFrameMapper

In [27]:
gen_df = suppressed_df.copy()

# numerical generalization: patient_age -> 10-year ranges
bins = list(range(0, 91, 10))
labels = ["{}-{}".format(b, b + 9) for b in bins[:-1]]
gen_df["patient_age"] = pd.cut(gen_df["patient_age"], bins=bins, labels=labels, right=False)

encoders = [(["gender"], LabelEncoder()), (["country"], LabelEncoder())]
mapper = DataFrameMapper(encoders, df_out=True)
new_cols = mapper.fit_transform(gen_df.copy())
gen_df = pd.concat([gen_df.drop(columns=["gender", "country"]), new_cols], axis="columns")

gen_df[["patient_age", "gender", "country", "virus_strain", "fatality"]].head(8)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,patient_age,gender,country,virus_strain,fatality
0,20-29,1,3,Sin Nombre,No
1,40-49,0,1,Sin Nombre,No
2,30-39,0,4,Seoul,No
3,40-49,1,0,Sin Nombre,No
4,50-59,1,4,Dobrava,No
5,50-59,1,7,Puumala,No
6,70-79,0,9,Dobrava,No
7,70-79,1,8,Sin Nombre,No


In [29]:
suppressed_df[["patient_age", "gender", "country", "virus_strain", "fatality"]].head(8)

,patient_age,gender,country,virus_strain,fatality
0,26,Male,Canada,Sin Nombre,No
1,41,Female,Bolivia,Sin Nombre,No
2,39,Female,Chile,Seoul,No
3,49,Male,Argentina,Sin Nombre,No
4,59,Male,Chile,Dobrava,No
5,53,Male,Peru,Puumala,No
6,70,Female,Uruguay,Dobrava,No
7,75,Male,USA,Sin Nombre,No


**explanation:** `patient_age` is now a band like "30-39" instead of an exact age, so a
single year can no longer pinpoint someone. `gender` and `country` are replaced by integer
codes (e.g. 0/1 for gender), hiding the literal category. anyone reading this version of the table will not know the precise values; this makes identifying an individual from these identifiers much harder.

## 4. perturbation (listings 7.1 and 7.2)
perturbation replaces records with random data that has the **same statistical properties**.
- for a **continuous** column, fit the best probability distribution to it
  (`best_fit_distribution`) and sample fresh values from that distribution.
- for a **categorical** column, sample new values with `np.random.choice` using the original
  value frequencies as probabilities.

we apply this to a continuous column (`temperature_celsius`) and a categorical one (`virus_strain`).

In [30]:
# loop candidate distributions, keep the best-fitting one.
def best_fit_distribution(data, bins=200):
    data = pd.Series(data).dropna().values
    y, x = np.histogram(data, bins=bins, density=True)
    x = (x[:-1] + x[1:]) / 2.0
    candidates = ['norm', 'gamma', 'lognorm', 'beta', 'expon', 'uniform', 'triang']
    best_name, best_params, best_sse = 'norm', (np.mean(data), np.std(data)), np.inf
    for name in candidates:
        dist = getattr(scipy.stats, name)
        try:
            params = dist.fit(data)
            pdf = dist.pdf(x, *params)
            sse = np.sum(np.power(y - pdf, 2.0))
            if np.isfinite(sse) and sse < best_sse:
                best_name, best_params, best_sse = name, params, sse
        except Exception:
            pass
    return best_name, best_params

In [31]:
# set up which columns to perturb
categorical = ['virus_strain']
continuous = ['temperature_celsius']

base = gen_df.copy()
unchanged = [c for c in list(base) if c not in categorical and c not in continuous]

# find the best-fit distribution for each continuous column
best_distributions = []
for col in continuous:
    name, params = best_fit_distribution(base[col], 200)
    best_distributions.append((name, params))
    print("best-fit distribution for {}: {}".format(col, name))

/usr/local/lib/python3.12/dist-packages/scipy/stats/_continuous_distns.py:801: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  a, b = optimize.fsolve(func, (1.0, 1.0))


best-fit distribution for temperature_celsius: lognorm


In [32]:
# listing 7.2: perturbation for both numerical and categorical values
def perturb_data(df, unchanged_cols, categorical_cols, continuous_cols,
                 best_distributions, n, seed=0):
    np.random.seed(seed)
    data = {}
    for col in categorical_cols: # categorical: resample by frequency
        counts = df[col].value_counts()
        data[col] = np.random.choice(list(counts.index),
                                     p=(counts / len(df)).values, size=n)
    for col, bd in zip(continuous_cols, best_distributions): # continuous: sample the fitted dist
        dist = getattr(scipy.stats, bd[0])
        data[col] = np.round(dist.rvs(size=n, *bd[1]), 1)
    for col in unchanged_cols: # pass everything else through as it is
        data[col] = df[col].values
    return pd.DataFrame(data, columns=unchanged_cols + categorical_cols + continuous_cols)

perturbed_df = perturb_data(base, unchanged, categorical, continuous,
                            best_distributions, n=len(base))
perturbed_df[["virus_strain", "temperature_celsius", "fatality", "hospitalization"]].head(8)

,virus_strain,temperature_celsius,fatality,hospitalization
0,Andes,23.5,No,Yes
1,Dobrava,32.4,No,Yes
2,Andes,14.3,No,No
3,Andes,25.1,No,No
4,Andes,14.7,No,No
5,Dobrava,29.2,No,Yes
6,Andes,16.9,No,Yes
7,Puumala,25.9,No,Yes


In [33]:
# show that perturbation preserved the statistics while changing individual values
print("original temperature_celsius -> mean {:.2f}, std {:.2f}".format(
    original_df["temperature_celsius"].mean(), original_df["temperature_celsius"].std()))
print("perturbed temperature_celsius -> mean {:.2f}, std {:.2f}".format(
    perturbed_df["temperature_celsius"].mean(), perturbed_df["temperature_celsius"].std()))
print()
print("original virus_strain proportions:")
print((original_df["virus_strain"].value_counts(normalize=True)).round(3).to_string())
print("\nperturbed virus_strain proportions:")
print((perturbed_df["virus_strain"].value_counts(normalize=True)).round(3).to_string())

original temperature_celsius -> mean 24.17, std 5.98
perturbed temperature_celsius -> mean 23.97, std 5.85

original virus_strain proportions:
virus_strain
Sin Nombre    0.216
Seoul         0.207
Andes         0.197
Dobrava       0.194
Puumala       0.187

perturbed virus_strain proportions:
virus_strain
Sin Nombre    0.218
Seoul         0.208
Puumala       0.203
Andes         0.186
Dobrava       0.184


**explanation:** every patient's `temperature_celsius` is now a value freshly drawn from the distribution that best fit the original column, and each `virus_strain` is resampled from the original strain frequencies. the per-row values no longer correspond to any real patient, but the column-level statistics (mean/std of temperature, the mix of strains) are preserved, so aggregate analysis still works while individual records are no longer genuine.

## 5. anatomization
anatomization keeps the original values but splits them into **two separate tables** - one
holding sensitive attributes and one holding quasi-identifiers - joined only by a shared group key, so the two can't be  linked back together to identify someone.

we split the original dataset into:
- a **quasi-identifier** table: country, region, patient_age, gender, report_date.
- a **sensitive-attribute** table: virus_strain, symptoms, hospitalization, fatality, recovery_days.

both get a `group_id` rather than the real `case_id`, and within each group the rows are shuffled so a row in one table can't be matched 1-to-1 to a row in the other.

In [34]:
anat_src = original_df.copy()

quasi_cols = ["country", "region", "patient_age", "gender", "report_date"]
sensitive_cols = ["virus_strain", "symptoms", "hospitalization", "fatality", "recovery_days"]

# assign each record to a small group (group of app. 50 here) and shuffle within group
group_size = 50
anat_src = anat_src.sample(frac=1, random_state=0).reset_index(drop=True)
anat_src["group_id"] = anat_src.index // group_size

# two released tables, each shuffled within its group so the link is broken
qi_table = (anat_src[["group_id"] + quasi_cols]
            .groupby("group_id", group_keys=False).apply(lambda g: g.sample(frac=1, random_state=1)))
sens_table = (anat_src[["group_id"] + sensitive_cols]
              .groupby("group_id", group_keys=False).apply(lambda g: g.sample(frac=1, random_state=2)))

print("quasi-identifier table:", qi_table.shape)
qi_table.head(6)

/tmp/ipykernel_8751/872830425.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .groupby("group_id", group_keys=False).apply(lambda g: g.sample(frac=1, random_state=1)))
/tmp/ipykernel_8751/872830425.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .groupby("group_id", group_keys=False).apply(lambda g: g.sample(frac=1, random_state=2)))


quasi-identifier table: (2000, 6)


,group_id,country,region,patient_age,gender,report_date
27,0,Uruguay,Jamesmouth,42,Male,18-01-2026
35,0,Brazil,Willisbury,35,Female,12-07-2025
40,0,Uruguay,Chapmanfort,47,Female,11-05-2026
38,0,Peru,Jacksonstad,55,Male,27-01-2026
2,0,Canada,Jonesshire,28,Female,22-04-2025
3,0,Bolivia,Johnsonstad,62,Male,09-04-2025


In [35]:
print("sensitive-attribute table:", sens_table.shape)
sens_table.head(6)

sensitive-attribute table: (2000, 6)


,group_id,virus_strain,symptoms,hospitalization,fatality,recovery_days
36,0,Puumala,"Shortness of Breath, Fever",No,No,16.0
47,0,Andes,"Shortness of Breath, Fever",Yes,No,41.0
28,0,Dobrava,"Fatigue, Nausea, Chills",No,No,30.0
9,0,Andes,"Headache, Vomiting, Fever",No,No,40.0
13,0,Puumala,"Headache, Vomiting, Fever",No,No,38.0
0,0,Dobrava,"Fever, Cough, Headache",No,No,21.0


**explanation:** the exact values are untouched - no noise, no generalization - but the record is split into two tables linked only by a coarse `group_id`, and rows are shuffled inside each group. so we can still learn, for a group, the *set* of ages/countries and the *set* of strains/outcomes, but we can't tell which specific patient in the group had which outcome. that broken linkage between quasi-identifiers and sensitive attributes is what protects records in the table while keeping both halves available for analysis.

## summary and insights

| technique | what it did here | effect |
|---|---|---|
| suppression | dropped `case_id`, `region` | removes direct identifiers entirely |
| generalization | `patient_age` -> 10-yr bands; encoded `gender`, `country` | exact values replaced by coarser ones |
| perturbation | resampled `temperature_celsius` (fitted dist) and `virus_strain` (frequencies) | rows become synthetic but column statistics are preserved |
| anatomization | split into quasi-identifier and sensitive tables, shuffled within groups | values kept, but the link between identity and sensitive info is broken |

each technique trades a different amount of utility for privacy: suppression and generalization lose detail permanently; perturbation keeps aggregate statistics but discards true per-row values; anatomization keeps every value but hides which values belong together. in practice they are combined (suppress -> generalize -> perturb) to sanitize a dataset before publishing it.

note: this is a sensitive topic (medical data); these techniques are exactly the kind used to let such data be shared for research without exposing the individuals in it.
